# 05 与远端冻结三状态逐日比较

本 Notebook 只做冻结后的诊断比较：将 03 号 Notebook 输出的 `date / three_state` 与用户显式指定的远端三状态基准文件按实际执行日对齐。

它不参与候选筛选、不改变冻结参数，也不是上传包的生产输入。上传包正式运行仍只需要 `COMPANY_SPOT_PATH`；只有运行本 Notebook 并设置 `REMOTE_THREE_STATE_PATH` 时，才读取外部基准文件。

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 240)
pd.set_option('display.max_colwidth', 60)

PACKAGE_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'src' / 'pool_registry.py').is_file()
)
sys.path.insert(0, str(PACKAGE_ROOT / 'src'))
from diagnostic_utils import compare_state_frames, load_event_signal
from runtime_paths import resolve_output_dir, resolve_remote_three_state_path

OUTPUT_DIR = resolve_output_dir(PACKAGE_ROOT)
GENERATED_PATH = Path(os.environ.get('EVENT_SIGNAL_CSV_PATH', str(OUTPUT_DIR / 'company_event_three_state_signal.csv'))).expanduser().resolve()
BASELINE_PATH = resolve_remote_three_state_path()
generated = load_event_signal(GENERATED_PATH)[['date', 'three_state']].copy()
print('生成文件:', GENERATED_PATH)
print('生成文件日期:', generated['date'].min().date(), '->', generated['date'].max().date(), '；行数:', len(generated))

print('远端基准文件:', BASELINE_PATH)
if not BASELINE_PATH.is_file():
    print('BASELINE_NOT_FOUND：路径不存在；不把缺失基准误报为一致。')
    display(pd.DataFrame([{'status': 'BASELINE_NOT_FOUND', 'generated_rows': len(generated), 'baseline_path': str(BASELINE_PATH)}]))
else:
    baseline_raw = pd.read_csv(BASELINE_PATH)
    if not {'date', 'three_state'}.issubset(baseline_raw.columns):
        raise ValueError(f'基准文件必须包含 date/three_state；实际列={list(baseline_raw.columns)}')
    baseline = baseline_raw[['date', 'three_state']].copy()
    baseline['date'] = pd.to_datetime(baseline['date'], errors='raise').dt.normalize()
    baseline['three_state'] = pd.to_numeric(baseline['three_state'], errors='raise').astype(int)
    if not baseline['three_state'].isin([-1, 0, 1]).all():
        raise ValueError('基准 three_state 含有 -1/0/1 之外的值')
    baseline = baseline.drop_duplicates('date', keep='last').sort_values('date').reset_index(drop=True)
    summary, cross, counts, mismatches = compare_state_frames(generated, baseline)
    print('比较摘要：')
    display(summary.round(6))
    print('共同日期上的状态列联表（行=生成，列=远端基准）：')
    display(cross)
    print('逐状态天数：')
    display(counts)
    common = generated.merge(baseline, on='date', how='inner', suffixes=('_generated', '_baseline'))
    common['period'] = np.select([common['date'].le(pd.Timestamp('2022-12-31')), common['date'].le(pd.Timestamp('2024-12-31'))], ['Development', 'Validation'], default='Test')
    period_compare = common.groupby('period', sort=False).agg(common_dates=('date', 'size'), matching_dates=('three_state_generated', lambda x: int(x.eq(common.loc[x.index, 'three_state_baseline']).sum()))).reset_index()
    period_compare['mismatch_dates'] = period_compare['common_dates'] - period_compare['matching_dates']
    period_compare['match_rate'] = period_compare['matching_dates'] / period_compare['common_dates']
    print('分周期一致率：')
    display(period_compare.round(6))
    if mismatches.empty:
        print('COMPARE_STATUS: 完全一致（共同 date 上 three_state 没有差异）')
    else:
        print(f'COMPARE_STATUS: 存在 {len(mismatches)} 个共同日期差异')
        display(mismatches[['date', 'generated_three_state', 'baseline_three_state', 'delta_generated_minus_baseline']])
    print('COMPARE_THREE_STATE_END')